# 04 · Đồng bộ slide và transcript

**Input:** Output 01 + 02 + 03. **Thiết bị:** CPU. **Output:** 04_bundle.

Kiểm tra checksum video và phiên bản phát hiện slide của hai nhánh. Gán mỗi từ
vào interval chứa midpoint, giữ occurrence lặp lại và previous/next IDs.
Span thiếu word timing được tham chiếu chung khi qua ranh giới; không ước lượng
thời gian từng từ. Gói kết quả chứa hình ảnh, vùng crop và audio theo slide.


## 1. Môi trường

Bật **Internet** để cài thư viện/tải model. Chạy mỗi notebook trong phiên Kaggle mới, với Input và GPU như ghi ở trên.


Chỉ dùng thư viện chuẩn Python.


## 2. Hàm xử lý

Các hàm nhúng trực tiếp; không cần tải file Python đi kèm. Mở ô thu gọn để xem hoặc chỉnh thuật toán.


In [ ]:
"""Shared artifact contract. Embedded into every notebook by build_notebooks.py."""

from pathlib import Path
import hashlib
import importlib.metadata
import json
import math
import re
import shutil
import subprocess
import threading
import unicodedata
from datetime import datetime, timezone

# transformers' from_pretrained builds models under a process-wide "meta device" context
# that is not thread-safe: two loads at once (OCR thread and main thread) fail with
# "Cannot copy out of meta tensor". Every model is loaded once per session, one at a time.
MODEL_LOAD_LOCK = threading.RLock()
_SHARED_MODELS = {}


def shared_model(key, loader):
    """Load a model once per session (reused by every video) and never two at once."""
    with MODEL_LOAD_LOCK:
        if key not in _SHARED_MODELS:
            _SHARED_MODELS[key] = loader()
        return _SHARED_MODELS[key]


SCHEMA_VERSION = "lecture-rag-v1"
PIPELINE_VERSION = "l25-v1"
STAGE_NAMES = {
    1: "01_slides", 2: "02_visual", 3: "03_audio",
    4: "04_bundle", 5: "05_index", 6: "06_retrieval",
}


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    temporary.replace(path)


def fingerprint(value):
    data = json.dumps(value, ensure_ascii=False, sort_keys=True, allow_nan=False)
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def relative_asset(root, name):
    root = Path(root).resolve()
    path = (root / name).resolve()
    if not path.is_relative_to(root):
        raise ValueError(f"Asset nằm ngoài thư mục kết quả: {name}")
    return path


def run_command(arguments):
    process = subprocess.run(
        [str(value) for value in arguments], capture_output=True, text=True,
        encoding="utf-8", errors="replace",
    )
    if process.returncode:
        raise RuntimeError(
            f"{arguments[0]} thất bại ({process.returncode}):\n{process.stderr[-3000:]}"
        )
    return process.stdout


def package_versions(names):
    versions = {}
    for name in names:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = "not-installed"
    return versions


def find_video(override=None, input_root="/kaggle/input"):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f"Không tìm thấy video: {path}")
        return path
    candidates = sorted(
        path for path in Path(input_root).rglob("*")
        if path.is_file()
        and path.suffix.lower() in {".mp4", ".mkv", ".mov", ".webm"}
        and path.stem.lower() in {"l25_v001", "tl25_v001"}
    )
    if len(candidates) != 1:
        raise ValueError(
            "Cần đúng một video L25_V001/TL25_V001. Add dataset "
            "thnhtrungnguynmif/l25a-lecture-videos-media hoặc đặt VIDEO_PATH. "
            f"Tìm thấy: {[str(path) for path in candidates]}"
        )
    return candidates[0]


def probe_video(path):
    if not shutil.which("ffprobe") or not shutil.which("ffmpeg"):
        raise RuntimeError("Cần FFmpeg và FFprobe trong PATH.")
    data = json.loads(run_command([
        "ffprobe", "-v", "error", "-show_format", "-show_streams",
        "-of", "json", str(path),
    ]))
    videos = [stream for stream in data["streams"] if stream["codec_type"] == "video"]
    audios = [stream for stream in data["streams"] if stream["codec_type"] == "audio"]
    if not videos:
        raise ValueError("Tệp không có luồng video.")
    info = data.get("format", {})
    origin = float(info.get("start_time", 0) or 0)
    duration = float(info.get("duration") or videos[0].get("duration") or 0)
    if not math.isfinite(duration) or duration <= 0:
        raise ValueError("Không đọc được thời lượng video hợp lệ.")
    return {
        "filename": Path(path).name,
        "sha256": file_sha256(path),
        "duration_sec": duration,
        "timeline_origin_sec": origin,
        "video_start_sec": float(videos[0].get("start_time", origin)),
        "audio_start_sec": float(audios[0].get("start_time", origin)) if audios else None,
        "has_audio": bool(audios),
        "width": int(videos[0]["width"]),
        "height": int(videos[0]["height"]),
        "clock": "original_presentation_timestamp_minus_format_start_time",
    }


def begin_stage(output_root, stage):
    directory = Path(output_root) / STAGE_NAMES[stage]
    directory.mkdir(parents=True, exist_ok=True)
    write_json(directory / "_RUNNING.json", {
        "stage": stage, "started_utc": datetime.now(timezone.utc).isoformat(),
    })
    return directory


def finish_stage(directory, stage, video_id, source, parameters, parents=(), extra=None):
    directory = Path(directory)
    files = {
        path.relative_to(directory).as_posix(): file_sha256(path)
        for path in sorted(directory.rglob("*"))
        if path.is_file()
        and path.name != "manifest.json"
        and not any(part.startswith("_") for part in path.relative_to(directory).parts)
        and path.suffix != ".tmp"
    }
    payload = {
        "schema_version": SCHEMA_VERSION,
        "pipeline_version": PIPELINE_VERSION,
        "stage": stage,
        "video_id": video_id,
        "source": source,
        "parameters": parameters,
        "parents": {str(parent["stage"]): parent["artifact_id"] for parent in parents},
        "files": files,
        **(extra or {}),
    }
    payload["artifact_id"] = fingerprint(payload)
    payload["created_utc"] = datetime.now(timezone.utc).isoformat()
    write_json(directory / "manifest.json", payload)
    (directory / "_RUNNING.json").unlink(missing_ok=True)
    print(f"Hoàn tất {STAGE_NAMES[stage]}: {directory}")
    return payload


def load_stage(directory, stage=None, video_id="L25_V001", verify=True):
    directory = Path(directory)
    if (directory / "_RUNNING.json").exists():
        raise ValueError(f"Bước xử lý chưa hoàn tất; cần chạy lại: {directory}")
    manifest = read_json(directory / "manifest.json")
    if manifest.get("schema_version") != SCHEMA_VERSION:
        raise ValueError(f"Schema không tương thích: {directory}")
    if stage is not None and manifest.get("stage") != stage:
        raise ValueError(f"Cần stage {stage}, nhận {manifest.get('stage')}: {directory}")
    if manifest.get("video_id") != video_id:
        raise ValueError(f"Sai video_id: {manifest.get('video_id')} != {video_id}")
    if manifest.get("pipeline_version") != PIPELINE_VERSION:
        raise ValueError(f"Sai pipeline_version: {directory}")
    signed = {key: value for key, value in manifest.items()
              if key not in {"artifact_id", "created_utc"}}
    if fingerprint(signed) != manifest["artifact_id"]:
        raise ValueError(f"Manifest đã thay đổi sau khi xuất: {directory}")
    if verify:
        for name, expected in manifest["files"].items():
            path = relative_asset(directory, name)
            if not path.is_file() or file_sha256(path) != expected:
                raise ValueError(f"Asset bị thiếu hoặc không khớp checksum: {path}")
    return manifest


def find_stage(stage, override=None, video_id="L25_V001",
               search_roots=("/kaggle/input", "/kaggle/working")):
    if override:
        directory = Path(override)
        return directory, load_stage(directory, stage, video_id)
    matches = []
    for root in map(Path, search_roots):
        if not root.exists():
            continue
        for path in root.rglob("manifest.json"):
            try:
                manifest = read_json(path)
            except (ValueError, OSError):
                continue
            if (manifest.get("schema_version") == SCHEMA_VERSION
                    and manifest.get("stage") == stage
                    and manifest.get("video_id") == video_id
                    and manifest.get("pipeline_version") == PIPELINE_VERSION):
                matches.append((path.parent, manifest))
    identities = {manifest.get("artifact_id") for _, manifest in matches}
    if not matches or len(identities) != 1:
        raise ValueError(
            f"Cần một phiên bản rõ ràng của {STAGE_NAMES[stage]}. "
            "Add Input → Notebook Output của bước trước, hoặc đặt STAGE_XX_DIR. "
            f"Các thư mục tìm được: {[str(path) for path, _ in matches]}"
        )
    directory = sorted(matches, key=lambda item: str(item[0]))[0][0]
    return directory, load_stage(directory, stage, video_id)


def require_same_source(*manifests):
    identities = {
        (item["video_id"], item["source"]["sha256"], item["pipeline_version"])
        for item in manifests
    }
    if len(identities) != 1:
        raise ValueError("Các nhánh không được tạo từ cùng video/pipeline.")


def require_parent(child, parent):
    require_same_source(child, parent)
    if child["parents"].get(str(parent["stage"])) != parent["artifact_id"]:
        raise ValueError(
            f"Stage {child['stage']} dùng phiên bản stage {parent['stage']} khác. "
            "Chạy lại nhánh với cùng kết quả phát hiện slide."
        )


def verify_intervals(intervals, duration, tolerance=0.03):
    if not intervals:
        raise ValueError("Danh sách interval rỗng.")
    ids = set()
    previous_end = 0.0
    for index, item in enumerate(intervals):
        start, end = float(item["start_sec"]), float(item["end_sec"])
        if not (math.isfinite(start) and math.isfinite(end)
                and 0 <= start < end <= duration + tolerance):
            raise ValueError(f"Interval không hợp lệ: {item}")
        if abs(start - previous_end) > tolerance:
            raise ValueError(f"Khoảng trống/chồng lấn ở interval {index}.")
        if item["segment_id"] in ids:
            raise ValueError("Trùng segment_id.")
        ids.add(item["segment_id"])
        previous_end = end
    if abs(previous_end - duration) > tolerance:
        raise ValueError("Interval cuối không bao phủ hết video.")


def owner_interval(intervals, time_sec):
    # Half-open intervals: a word exactly at a boundary belongs to the next one.
    for item in intervals:
        if item["start_sec"] <= time_sec < item["end_sec"]:
            return item
    if intervals and math.isclose(time_sec, intervals[-1]["end_sec"], abs_tol=1e-6):
        return intervals[-1]
    raise ValueError(f"Timestamp nằm ngoài lecture: {time_sec}")


def copy_stage_assets(source, destination, manifest):
    for name in manifest["files"]:
        target = relative_asset(destination, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(relative_asset(source, name), target)


def normalize_search(text):
    text = unicodedata.normalize("NFD", text.casefold().replace("đ", "d"))
    return "".join(char for char in text if not unicodedata.combining(char))


def timestamp_label(seconds):
    seconds = max(0, int(seconds))
    hours, seconds = divmod(seconds, 3600)
    minutes, seconds = divmod(seconds, 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}" if hours else f"{minutes:02}:{seconds:02}"


def unique_records(records, key):
    result = {}
    for record in records:
        if record[key] in result:
            raise ValueError(f"Trùng {key}: {record[key]}")
        result[record[key]] = record
    return result


def read_pcm_audio(path, sample_rate=16000):
    import numpy as np
    import wave

    with wave.open(str(path), "rb") as stream:
        if (stream.getframerate() != sample_rate or stream.getnchannels() != 1
                or stream.getsampwidth() != 2):
            raise ValueError("Audio phải là PCM signed 16-bit, mono, 16 kHz.")
        return np.frombuffer(stream.readframes(stream.getnframes()), dtype="<i2").copy()


def save_pcm_clip(path, pcm, start_sample, end_sample, sample_rate=16000):
    import wave

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(path), "wb") as stream:
        stream.setnchannels(1)
        stream.setsampwidth(2)
        stream.setframerate(sample_rate)
        stream.writeframes(pcm[start_sample:end_sample].astype("<i2").tobytes())


In [ ]:
"""Clean slide OCR before indexing and correct lecture terms the ASR misheard, using the
words printed on the slides of the same video (stage 04; raw outputs stay untouched).

OCR: lines repeated on most slides (channel logo, lecturer name, school, page numbers),
"[Unreadable]" markers and figure regions that only describe the lecturer or a logo are
noise for search. ASR: Whisper often hears a domain term with a confusable consonant or
final ("xác xuất" for "xác suất", "số chẳng" for "số chẵn"); when the slides print a
near-identical term and never the heard one, the heard words are replaced. Corrections
keep the word timing and the original text (text_asr).
"""

UNREADABLE = re.compile(r"\[(?:unreadable|non-text)\]", re.I)
# Unlimited-OCR box markup that leaks into the text, e.g. a cut-off "<|det|>header" at the end
LEAKED_TAG = re.compile(r"<\|det\|>\s*[A-Za-z_]*\s*(?:\[[\d,\s]*\]?)?|<\|/?\w{1,20}\|>")
NOISE_REGION_WORDS = ("giáo viên", "giảng viên", "người", "logo", "trang trí", "teacher", "presenter",
                      "decorative", "khuôn mặt", "chân dung")
# Onsets Whisper confuses in Vietnamese (s/x, ch/tr); a final c/t or n/ng may differ in the same
# syllable. Tone-only differences are never corrected: slide OCR drops tones more often than ASR.
ONSETS = [("tr", "ch"), ("s", "x")]
FINALS = [("ng", "n"), ("c", "t")]
OCR_FIXES = {"ǎ": "ă", "Ǎ": "Ă"}   # Unlimited-OCR writes a caron for the breve


def strip_tones(text):
    text = unicodedata.normalize("NFD", text.lower())
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return text.replace("đ", "d")


def line_key(line):
    """Comparable form of an OCR line: no accents or punctuation (digits kept: "Câu 1" != "Câu 2")."""
    return re.sub(r"[^a-z0-9]+", "", strip_tones(line))


def repeated_lines(texts, min_share=0.5, min_slides=3):
    """Line keys that appear on at least min_share of the slides (and >= min_slides)."""
    counts, slides = {}, 0
    for text in texts:
        keys = {line_key(line) for line in (text or "").splitlines() if line_key(line)}
        if not keys:
            continue
        slides += 1
        for key in keys:
            counts[key] = counts.get(key, 0) + 1
    need = max(min_slides, math.ceil(min_share * slides))
    return {key for key, count in counts.items() if count >= need}


# Unlimited-OCR sometimes loops: L25_V005 seg 9 (lecturer + a small inset slide) came back as one
# formula line 269 times. Runs of LOOP_RUN or more are kept once; two identical answer options stay.
LOOP_RUN = 3
LOOP_PHRASE = re.compile(r"(\S.{5,}?)(?:\s*\1){%d,}" % (LOOP_RUN - 1))


def collapse_loops(lines, min_run=LOOP_RUN):
    """Keep one line of each run of >= min_run consecutive lines with the same line_key."""
    kept, index = [], 0
    while index < len(lines):
        key, end = line_key(lines[index]), index + 1
        while key and end < len(lines) and line_key(lines[end]) == key:
            end += 1
        kept.extend(lines[index:end] if end - index < min_run else lines[index:index + 1])
        index = end
    return kept


def collapse_phrase_loop(line):
    """A phrase with letters repeated back to back >= LOOP_RUN times in one line is kept once
    ("0 0 0 0" in a table row has no letters and stays)."""
    return LOOP_PHRASE.sub(lambda match: match.group(1) if re.search(r"[^\W\d_]", match.group(1))
                           else match.group(0), line)


def clean_ocr_text(text, repeated=frozenset()):
    kept = []
    for line in collapse_loops((text or "").splitlines()):
        line = collapse_phrase_loop(UNREADABLE.sub("", LEAKED_TAG.sub("", line))).strip()
        key = line_key(line)
        if not line or key in repeated:
            continue
        if len(line) <= 3 and not re.search(r"[A-Za-zÀ-ỹ]{2}", line):   # page numbers, bullets
            continue
        for bad, good in OCR_FIXES.items():
            line = line.replace(bad, good)
        kept.append(line)
    return "\n".join(kept)


def lines_new_since(previous_text, text, min_key=12):
    """Lines of text not already on the previous slide (same line_key), and the share of long
    lines that were. A slide shown again after an ad, a reveal split in two or a document
    page moved down a little repeats ~32 % of its lines on L25; the first line (title) stays."""
    seen = {line_key(line) for line in (previous_text or "").splitlines()}
    lines = (text or "").splitlines()
    long_keys = [line_key(line) for line in lines if len(line_key(line)) >= min_key]
    share = sum(key in seen for key in long_keys) / len(long_keys) if long_keys else 0.0
    kept = [line for index, line in enumerate(lines)
            if index == 0 or len(line_key(line)) < min_key or line_key(line) not in seen]
    return "\n".join(kept), round(share, 3)


def noise_region(region):
    """A figure region whose model description is only about the lecturer, a logo or decoration."""
    if region.get("kind") not in {"diagram", "figure", "image"}:
        return False
    text = f"{region.get('text') or ''} {region.get('description') or ''}".lower()
    has_numbers = bool(re.search(r"\d", region.get("text") or ""))
    return any(word in text for word in NOISE_REGION_WORDS) and not has_numbers


def clean_regions(regions):
    """Marks noise and duplicate regions (same text on the same slide) with noise=True."""
    seen = set()
    out = []
    for region in regions or []:
        region = dict(region)
        key = line_key(f"{region.get('text') or ''}{region.get('description') or ''}")
        region["noise"] = noise_region(region) or (key in seen and bool(key))
        seen.add(key)
        out.append(region)
    return out


def syllables(text):
    return re.findall(r"\w+", (text or "").lower())


def split_syllable(s):
    """(onset, rest) of a toneless syllable for the confusable onsets."""
    for onset in ("tr", "ch", "s", "x"):
        if s.startswith(onset):
            return onset, s[len(onset):]
    return "", s


def confusable(heard, printed):
    """True if a heard syllable can be a mishearing of the printed one: identical, or the onset
    is s/x or ch/tr swapped (rest equal up to tone and a c/t or n/ng final)."""
    if heard == printed:
        return True
    a, b = strip_tones(heard), strip_tones(printed)
    (oa, ra), (ob, rb) = split_syllable(a), split_syllable(b)
    if not oa or not ob or oa == ob or {oa, ob} not in ({"s", "x"}, {"ch", "tr"}):
        return False
    if ra == rb:
        return True
    for x, y in FINALS:
        for p, q in ((x, y), (y, x)):
            if ra.endswith(p) and ra[:-len(p)] + q == rb:
                return True
    return False


def ocr_vocabulary(texts, sizes=(2, 3)):
    """Counts of the syllable n-grams printed on the slides (accented, lower case)."""
    vocab = {}
    for text in texts:
        for line in clean_ocr_text(text).splitlines():
            syl = syllables(line)
            for n in sizes:
                for i in range(len(syl) - n + 1):
                    gram = tuple(syl[i:i + n])
                    if all(s.isalpha() for s in gram):
                        vocab[gram] = vocab.get(gram, 0) + 1
    return vocab


def correct_words(words, vocab, min_count=2):
    """Replace misheard n-grams of ASR words by the near-identical n-gram printed on the slides.
    words: [{'text', ...}] in time order. Returns (new words, corrections)."""
    words = [dict(w) for w in words]
    by_size = {}
    for gram, count in vocab.items():
        if count >= min_count:
            by_size.setdefault(len(gram), []).append(gram)
    corrections = []
    i = 0
    while i < len(words):
        done = False
        for n in (2, 3):
            window = words[i:i + n]
            if len(window) < n:
                continue
            heard = tuple(syllables(w["text"])[0] if syllables(w["text"]) else "" for w in window)
            if "" in heard or heard in vocab:
                continue
            matches = [gram for gram in by_size.get(n, ())
                       if gram != heard and all(confusable(h, g) for h, g in zip(heard, gram))]
            if len({m for m in matches}) != 1:
                continue
            gram = matches[0]
            for word, printed in zip(window, gram):
                if syllables(word["text"])[0] != printed:
                    word.setdefault("text_asr", word["text"])
                    word["text"] = re.sub(r"\w+", lambda m: printed.capitalize() if m.group(0)[0].isupper() else printed,
                                          word["text"], count=1)
            corrections.append({"at_sec": window[0]["start_sec"], "heard": " ".join(heard), "fixed": " ".join(gram)})
            i += n
            done = True
            break
        if not done:
            i += 1
    return words, corrections


In [ ]:
"""Join the two branches without estimating slide times or inventing word timing."""


def words_to_text(words):
    text = " ".join(word["text"].strip() for word in words)
    text = re.sub(r"\s+([,.;:!?%)\]])", r"\1", text)
    return re.sub(r"([(\[])\s+", r"\1", text).strip()


def words_to_spans(words, segment_id, maximum_sec=18.0, maximum_words=70):
    spans, group = [], []

    def append_group():
        spans.append({
            "span_id": f"{segment_id}:speech_{len(spans):04d}",
            "start_sec": min(word["start_sec"] for word in group),
            "end_sec": max(word["end_sec"] for word in group),
            "text": words_to_text(group),
            "word_ids": [word["word_id"] for word in group],
            "timing_precision": "word",
        })

    for word in words:
        if group and (word["end_sec"] - group[0]["start_sec"] > maximum_sec
                      or len(group) >= maximum_words):
            append_group()
            group = []
        group.append(word)
        if re.search(r"[.!?]$", word["text"]) and len(group) >= 4:
            append_group()
            group = []
    if group:
        append_group()
    return spans


def align_records(intervals, visuals, transcript, duration):
    verify_intervals(intervals, duration)
    visual_by_segment = unique_records(visuals, "segment_id")
    expected = {item["segment_id"] for item in intervals}
    if set(visual_by_segment) != expected:
        raise ValueError("Nhánh OCR phải có đúng một record cho mỗi visual interval.")
    words = unique_records(transcript["words"], "word_id")
    owned = {item["segment_id"]: [] for item in intervals}
    for word in sorted(words.values(), key=lambda item: (item["start_sec"], item["end_sec"])):
        start, end = word["start_sec"], word["end_sec"]
        if not (math.isfinite(start) and math.isfinite(end) and 0 <= start < end <= duration):
            raise ValueError(f"Word timing không hợp lệ: {word['word_id']}")
        interval = owner_interval(intervals, (start + end) / 2)
        owned[interval["segment_id"]].append(word)
    clips = unique_records(transcript["slide_audio_clips"], "segment_id")
    if transcript["status"] == "completed" and set(clips) != expected:
        raise ValueError("Thiếu audio clip để đối chiếu từng interval.")
    fallback = unique_records(transcript.get("fallback_spans", []), "span_id")
    for span in fallback.values():
        if not (0 <= span["start_sec"] < span["end_sec"] <= duration):
            raise ValueError(f"Fallback span không hợp lệ: {span['span_id']}")

    segments = []
    for index, interval in enumerate(intervals):
        segment_id = interval["segment_id"]
        visual = visual_by_segment[segment_id]
        spans = words_to_spans(owned[segment_id], segment_id)
        coarse_refs = [span["span_id"] for span in fallback.values()
                       if span["start_sec"] < interval["end_sec"]
                       and span["end_sec"] > interval["start_sec"]]
        segments.append({
            "segment_id": segment_id, "sequence": index,
            "start_sec": interval["start_sec"], "end_sec": interval["end_sec"],
            "previous_segment_id": intervals[index - 1]["segment_id"] if index else None,
            "next_segment_id": intervals[index + 1]["segment_id"]
            if index + 1 < len(intervals) else None,
            "content_kind": visual["content_kind"],
            # ad filter in "flag" mode: kept and read, left out of the index only (stage 05)
            **({"ad_flagged": interval["ad_flagged"], "ad_score": interval.get("ad_score")}
               if "ad_flagged" in interval else {}),
            "visual": {
                **visual,
                "image_path": interval["image_path"],
                "keyframe_time_sec": interval["keyframe_time_sec"],
            },
            "speech": {
                "text": words_to_text(owned[segment_id]), "spans": spans,
                "word_ids": [word["word_id"] for word in owned[segment_id]],
                "fallback_span_ids": coarse_refs,
                "status": "available" if spans or coarse_refs else
                ("no_audio" if transcript["status"] == "no_audio" else "no_speech_detected"),
            },
            "audio": clips.get(segment_id),
        })
    assigned = [word_id for segment in segments for word_id in segment["speech"]["word_ids"]]
    if len(assigned) != len(words) or set(assigned) != set(words):
        raise ValueError("Có từ bị mất hoặc bị gán lặp trong bước đồng bộ.")
    return segments


def word_pairs(text):
    words = re.findall(r"\w+", (text or "").lower())
    return {f"{a} {b}" for a, b in zip(words, words[1:])}


# Words that promote rather than teach. A course ad shares the subject of the lesson, so the
# topic score alone can call it relevant; these terms are counted separately (content filter B).
AD_TERMS = re.compile(r"đăng ký|ưu đãi|học phí|hotline|khóa học|khoá học|giảm giá|khuyến mãi|tuyển sinh|"
                      r"liên hệ|link|subscribe|sponsor|discount|enroll|sign up|promo", re.I)


def speech_windows(words, size=45.0, step=15.0):
    """[{start_sec, end_sec, text}] windows of the transcript words (time order)."""
    if not words:
        return []
    end, windows, start = words[-1]["end_sec"], [], 0.0
    while start < end:
        inside = [w for w in words if start <= (w["start_sec"] + w["end_sec"]) / 2 < start + size]
        if inside:
            windows.append({"start_sec": start, "end_sec": min(start + size, end), "text": words_to_text(inside)})
        start += step
    return windows


def speech_relevance(windows, embed, reference_texts=(), trim=0.25):
    """Score each speech window "belongs to this lecture": cosine to the lecture's topic centre,
    standardised inside the video (median / MAD), so no threshold is shared across channels.
    embed(texts) -> unit vectors. The centre ignores the trim share of least typical windows
    (ads, chit-chat) and also averages reference_texts (the slide text). Adds topic_score,
    topic_z and ad_terms to every window (in place) and returns them."""
    import numpy as np

    if not windows:
        return windows
    vectors = np.asarray(embed([w["text"] for w in windows]), dtype="float32")
    references = np.asarray(embed(list(reference_texts)), dtype="float32") if reference_texts else None
    centre = vectors.mean(0)
    for _ in range(2):   # re-centre without the least typical windows
        scores = vectors @ (centre / (np.linalg.norm(centre) + 1e-9))
        keep = scores >= np.quantile(scores, trim)
        pool = vectors[keep] if references is None else np.vstack([vectors[keep], references])
        centre = pool.mean(0)
    scores = vectors @ (centre / (np.linalg.norm(centre) + 1e-9))
    median = float(np.median(scores))
    mad = float(np.median(np.abs(scores - median))) * 1.4826 or 1e-6
    for window, score in zip(windows, scores):
        window["topic_score"] = round(float(score), 4)
        window["topic_z"] = round((float(score) - median) / mad, 3)
        window["ad_terms"] = len(AD_TERMS.findall(window["text"]))
    return windows


def review_ad_speech(segments, threshold, min_pairs=20):
    """Stage 01 cuts ads from the picture alone; a lesson stretch it cut by mistake would lose
    its speech from the transcript and the index. An ad segment whose word pairs mostly come
    back in the rest of the lecture or on its slides is marked ad_suspect: its speech is kept.
    L25 (9 videos, 20261002 run): 27 real ads overlap 0.022-0.246; 110 runs of lesson slides
    (40-85 s) relabelled as ad overlap >= 0.207 (median 0.44). Threshold 0.25: 0/27 ads flagged,
    103/110 runs kept; 0.20: 1/27 and 110/110."""
    lesson = set()
    for segment in segments:
        if segment["content_kind"] != "ad":
            visual = segment.get("visual") or {}
            lesson |= word_pairs(segment["speech"]["text"])
            lesson |= word_pairs(visual.get("ocr_text_clean", visual.get("ocr_text", "")))
    for segment in segments:
        if segment["content_kind"] != "ad":
            continue
        pairs = word_pairs(segment["speech"]["text"])
        overlap = len(pairs & lesson) / len(pairs) if len(pairs) >= min_pairs else None
        segment["ad_review"] = {"lesson_overlap": None if overlap is None else round(overlap, 4),
                                "word_pairs": len(pairs), "threshold": threshold}
        segment["ad_suspect"] = overlap is not None and overlap >= threshold
    return segments


def run_alignment(stage01_dir, stage02_dir, stage03_dir, output_root,
                  video_id="L25_V001", ad_suspect_overlap=None, speech_embed=None):
    """ad_suspect_overlap: None (off) or a threshold for review_ad_speech().
    speech_embed: None (off) or embed(texts) -> unit vectors (E5): every segment gets
    speech_topic = the lowest topic_z of the speech windows it overlaps (speech_relevance();
    recorded only, no decision yet) and speech_relevance.json keeps the windows."""
    paths = [Path(stage01_dir), Path(stage02_dir), Path(stage03_dir)]
    parents = [load_stage(path, index + 1, video_id) for index, path in enumerate(paths)]
    require_parent(parents[1], parents[0])
    independent_asr = "1" not in parents[2]["parents"]
    if independent_asr:
        # Stage 03 ran in parallel with stage 01 from the same video file.
        require_same_source(parents[2], parents[0])
    else:
        require_parent(parents[2], parents[0])
    intervals = read_json(paths[0] / "intervals.json")
    visuals = read_json(paths[1] / "visual.json")
    transcript = read_json(paths[2] / "transcript.json")
    # Clean slide OCR (lines repeated on most slides, [Unreadable], lecturer/logo regions) and fix
    # lecture terms Whisper misheard using the words printed on the slides (textclean.py).
    lesson = [visual for visual in visuals if visual.get("content_kind") != "ad"]
    repeated = repeated_lines([visual.get("ocr_text", "") for visual in lesson])
    for visual in visuals:
        visual["ocr_text_clean"] = clean_ocr_text(visual.get("ocr_text", ""), repeated)
        lines = [line for line in visual["ocr_text_clean"].splitlines() if len(line.strip()) >= 12]
        visual["title_clean"] = (lines[0][:160] if lines and visual.get("content_kind") in {"slide", "mixed", "board"}
                                 else "")
        visual["regions"] = clean_regions(visual.get("regions", []))
    previous = ""
    for visual in visuals:   # visual.json follows the interval order
        if visual.get("content_kind") == "ad":
            continue
        visual["ocr_text_new"], visual["ocr_repeat_share"] = lines_new_since(previous, visual["ocr_text_clean"])
        previous = visual["ocr_text_clean"] or previous
    vocabulary = ocr_vocabulary([visual["ocr_text_clean"] for visual in lesson])
    transcript["words"], asr_corrections = correct_words(
        sorted(transcript["words"], key=lambda word: (word["start_sec"], word["end_sec"])), vocabulary)
    directory = begin_stage(output_root, 4)
    audio_root = paths[2]
    if independent_asr and transcript["status"] == "completed":
        pcm = read_pcm_audio(paths[2] / "audio/lecture_16k.wav")
        transcript["slide_audio_clips"] = []
        for interval in intervals:
            start_sample = round(interval["start_sec"] * 16000)
            end_sample = min(len(pcm), round(interval["end_sec"] * 16000))
            name = f"slide_audio/seg_{interval['sequence']:05d}.wav"
            save_pcm_clip(directory / name, pcm, start_sample, end_sample)
            transcript["slide_audio_clips"].append({
                "segment_id": interval["segment_id"], "path": name,
                "start_sec": start_sample / 16000, "end_sec": end_sample / 16000,
            })
        audio_root = directory
    segments = align_records(intervals, visuals, transcript, parents[0]["source"]["duration_sec"])
    if ad_suspect_overlap is not None:
        review_ad_speech(segments, ad_suspect_overlap)
    windows = []
    if speech_embed is not None and transcript["words"]:
        slides = [visual["ocr_text_clean"] for visual in lesson if len(visual["ocr_text_clean"]) >= 40][:40]
        windows = speech_relevance(speech_windows(transcript["words"]), speech_embed, slides)
        for segment in segments:
            near = [w for w in windows if w["start_sec"] < segment["end_sec"] and w["end_sec"] > segment["start_sec"]]
            segment["speech_topic"] = ({"z": min(w["topic_z"] for w in near),
                                        "ad_terms": sum(w["ad_terms"] for w in near)} if near else None)
    for interval in intervals:
        name = interval["image_path"]
        target = relative_asset(directory, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(relative_asset(paths[0], name), target)
    for visual in visuals:
        if visual.get("content_image_path"):
            name = visual["content_image_path"]
            target = relative_asset(directory, name)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(relative_asset(paths[1], name), target)
        for region in visual["regions"]:
            name = region["image_path"]
            target = relative_asset(directory, name)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(relative_asset(paths[1], name), target)
    for clip in transcript["slide_audio_clips"] if audio_root != directory else ():
        name = clip["path"]
        target = relative_asset(directory, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(relative_asset(audio_root, name), target)

    write_json(directory / "segments.json", segments)
    if windows:
        write_json(directory / "speech_relevance.json", windows)
    write_json(directory / "asr_corrections.json", asr_corrections)
    write_json(directory / "words.json", transcript["words"])
    write_json(directory / "fallback_spans.json", transcript["fallback_spans"])
    write_json(directory / "asr_windows.json", transcript["windows"])
    # Compatible with the current app after mapping video_id to its uploaded ID.
    # Speech inside ad segments stays in segments.json but is not shown as lecture transcript,
    # unless review_ad_speech() found it sounds like the lesson (ad_suspect).
    app_spans = [span for segment in segments if segment["content_kind"] != "ad" or segment.get("ad_suspect")
                 for span in segment["speech"]["spans"]]
    app_spans += transcript["fallback_spans"]
    app_transcript = {
        "video_id": video_id, "language": transcript["language"],
        "segments": [
            {"start": span["start_sec"], "end": span["end_sec"], "text": span["text"]}
            for span in sorted(app_spans, key=lambda item: item["start_sec"])
        ],
    }
    write_json(directory / "transcript_app.json", app_transcript)
    manifest = finish_stage(
        directory, 4, video_id, parents[0]["source"],
        {"word_ownership": "midpoint_in_half_open_interval",
         "cross_boundary_fallback": "shared_reference_without_fabricated_word_times",
         "ocr_cleaning": {"repeated_lines": sorted(repeated)},
         "asr_correction": "slide_vocabulary_onset_s_x_ch_tr",
         **({"ad_suspect_overlap": ad_suspect_overlap} if ad_suspect_overlap is not None else {})},
        parents=parents,
        extra={"segment_count": len(segments), "word_count": len(transcript["words"]),
               "ad_segment_count": sum(segment["content_kind"] == "ad" for segment in segments),
               "ad_flagged_count": sum(bool(segment.get("ad_flagged")) for segment in segments),
               "asr_correction_count": len(asr_corrections),
               "noise_region_count": sum(region.get("noise", False) for visual in visuals for region in visual["regions"]),
               "ad_word_count": sum(len(segment["speech"]["word_ids"]) for segment in segments
                                    if segment["content_kind"] == "ad"),
               **({"ad_suspect_count": sum(bool(segment.get("ad_suspect")) for segment in segments)}
                  if ad_suspect_overlap is not None else {}),
               "bundle_status": "synchronized"},
    )
    return directory, manifest, segments


## 3. Cấu hình

Đường dẫn None sẽ được tìm trong Input. Nếu có nhiều phiên bản, điền đường dẫn rõ ràng để tránh chọn nhầm.


In [ ]:
from pathlib import Path

VIDEO_ID = 'L25_V001'
OUTPUT_ROOT = Path('/kaggle/working/lecture_rag') / VIDEO_ID

STAGE_01_DIR = None
STAGE_02_DIR = None
STAGE_03_DIR = None


## 4. Thực thi


In [ ]:
stage01_dir, _ = find_stage(1, STAGE_01_DIR, VIDEO_ID)
stage02_dir, _ = find_stage(2, STAGE_02_DIR, VIDEO_ID)
stage03_dir, _ = find_stage(3, STAGE_03_DIR, VIDEO_ID)
output_dir, manifest, segments = run_alignment(
    stage01_dir, stage02_dir, stage03_dir, OUTPUT_ROOT, VIDEO_ID,
)
print('Số đoạn đồng bộ:', len(segments), '| Số từ:', manifest['word_count'])


## 5. Kiểm tra / sử dụng kết quả


In [ ]:
from IPython.display import Image, display
for segment in segments[:3]:
    print(segment['segment_id'], segment['start_sec'], segment['end_sec'])
    display(Image(filename=str(output_dir / segment['visual']['image_path']), width=850))
    print('Loại:', segment['content_kind'])
    print('OCR:', segment['visual']['ocr_text'][:600])
    print('Lời giảng:', segment['speech']['text'][:1200])
    print('Trạng thái:', segment['speech']['status'])


## Tiếp theo

Save Version; notebook **05** chỉ cần Output này.
transcript_app.json tương thích format transcript backend sau khi ánh xạ
video_id sang ID lúc upload. Notebook không tự sửa dữ liệu của web app.
